# Silver (XAGUSD) 5-minute mock test: MLP template (warm colours (red, orange, navy))
**Multi-Layer Perceptron.** Stages 1 to 4 follow your Week 6 and Week 7 notebooks. Stages 5 to 9 follow the **Week 4** notebook structure.

**How to use**
1. Put the two CSV files in the same folder as this notebook.
2. Read the question and change only the numbers on the lines marked `# CHANGE`: the file names, the bar size (`'0 days 00:05:00'`), `0.8`, `window_size=5` with both `range(5)`, `N`, `n_small`, `test_size` and the model numbers.
3. Run the cells from top to bottom, **each cell once**, and leave the outputs on screen.

| Marks | Marking scheme item | Stage |
|---|---|---|
| 10 | Ask and Bid combined in one DataFrame | 1 |
| 10 | Preprocessing, graphs and histograms | 2 |
| 10 | Correct normalisation | 3 |
| 10 | Training and testing split | 4 |
| 10 | Neural network architecture | 5 |
| 20 | Neural network trained | 6 |
| 10 | Graph of the training process | 7 |
| 10 | MSE and MAE on the test data | 8 |
| 10 | Additional result graphs | 9 |

**Names used in this template**

| Group | Names |
|---|---|
| Bid prices and volume | `bid_o`, `bid_h`, `bid_l`, `bid_c`, `bid_v` |
| Ask prices and volume | `ask_o`, `ask_h`, `ask_l`, `ask_c`, `ask_v` |
| Spreads (Ask minus Bid) | `gap_o`, `gap_h`, `gap_l`, `gap_c`, `vol_gap`, `vol_gap_abs` |
| Time | `stamp` (text), `ts` (date-time), `ts_prev`, `ts_gap` |
| New day and week markers | `day_start`, `week_start` |
| Answers | `target_hi`, `target_lo` |
| Other | `ask_df`, `bid_df`, `both`, `feat_df`, `ans_df`, `feat_np`, `ans_np`, `tensor_x`, `p_hi`, `p_lo` |

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
%matplotlib inline

from matplotlib import*
import seaborn as sns

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
np.random.seed(19)
tf.random.set_seed(19)

In [ ]:
import datetime

## Stage 1: Combine the Ask and Bid time series into one DataFrame (10 marks)
**What you can change:** the two file names inside `pd.read_csv(...)`. Nothing else.

In [ ]:
ask_df = pd.read_csv('XAGUSD_5 Mins_Ask_2023.01.01_2023.06.30-1.csv')   # CHANGE: file name
bid_df = pd.read_csv('XAGUSD_5 Mins_Bid_2023.01.01_2023.06.30-1.csv')   # CHANGE: file name

In [ ]:
print(ask_df.head(3))
print(ask_df.tail(3))

In [ ]:
print(bid_df.head(3))
print(bid_df.tail(3))

In [ ]:
ask_df.info()

In [ ]:
bid_df.info()

In [ ]:
both = bid_df.merge(ask_df, left_on='Time (UTC)', right_on='Time (UTC)', how='outer')
both

In [ ]:
both.info()

In [ ]:
both.columns = ['stamp', 'bid_o', 'bid_h', 'bid_l', 'bid_c', 'bid_v',
                  'ask_o', 'ask_h', 'ask_l', 'ask_c', 'ask_v']

In [ ]:
print(both.shape)
print(both.isna().sum().sum(), 'missing values')

In [ ]:
file_obj2 = open('silver_merged.csv', 'w')
both.to_csv('silver_merged.csv', encoding='utf-8', index=False)
file_obj2.close()

In [ ]:
df = pd.read_csv('silver_merged.csv', low_memory=False, sep=',')

## Stage 2: Preprocessing, graphs and histograms (10 marks)
Delta columns (Ask minus Bid), time to date-time, price and volume charts, then the new day and new week markers.

**What you can change:**
- `bins=100`, the column names, the titles and the colours in the charts.
- `iloc[0:70]` in the candlestick chart: the rows to show.
- `'0 days 00:05:00'` (two cells) and `"2 days 01:05:00"` depend on the bar size. For 1-minute data use `'0 days 00:01:00'` and `"2 days 01:01:00"`; for 15-minute data use `00:15:00` and `01:15:00`.

In [ ]:
df['vol_gap'] = df['ask_v'] - df['bid_v']
df['vol_gap_abs'] = (df['ask_v'] - df['bid_v']).abs()

In [ ]:
df['gap_o'] = df['ask_o'] - df['bid_o']
df['gap_h'] = df['ask_h'] - df['bid_h']
df['gap_l'] = df['ask_l'] - df['bid_l']
df['gap_c'] = df['ask_c'] - df['bid_c']

In [ ]:
data = df.drop(['ask_o', 'ask_h', 'ask_l', 'ask_c'], axis=1)

In [ ]:
data['ts'] = pd.to_datetime(data['stamp'], utc=True)

In [ ]:
data = data.drop(['stamp'], axis=1)

In [ ]:
data.info()

In [ ]:
plt.figure(figsize=(15,4))
plt.plot(data['bid_c'], color='firebrick', linewidth=1.5)
plt.title('Silver close price, 2023', size=16)
plt.xlabel('N 5-minute bars')
plt.ylabel('Close price')
plt.grid(True, color='lightgray', linestyle='--')
plt.show()

In [ ]:
data_chart = data.set_index('ts', inplace=False)

In [ ]:
data_chart = data_chart.drop(['ask_v', 'vol_gap', 'vol_gap_abs',
                              'gap_o', 'gap_h', 'gap_l', 'gap_c'], axis=1)

In [ ]:
import mplfinance as mpf

data_chart.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
mc = mpf.make_marketcolors(up='darkorange', down='navy', edge='inherit', wick='inherit', volume='inherit')
my_style = mpf.make_mpf_style(base_mpf_style='charles', marketcolors=mc)
mpf.plot(data_chart.iloc[0:70], type='candle', style=my_style, title='Silver Candlestick Chart', volume=True)

In [ ]:
data_chart = []

In [ ]:
plt.figure(num=1, figsize=(14,7))
plt.hist(data['bid_c'], bins=100, color='gold', edgecolor='darkorange')
plt.title('Close (Bid) price distribution', size=16)
plt.ylabel('Numbers', size=12)
plt.xlabel('Close (Bid) price', size=12)
plt.show()

In [ ]:
plt.figure(num=1, figsize=(14,7))
data['bid_v'].plot.kde(color='crimson', linewidth=2)
plt.title('Volume (Bid) Density', size=16)
plt.ylabel('Density', size=12)
plt.xlabel('Volume (Bid)', size=12)
plt.grid(True, color='lightgray', linestyle='--')
plt.show()

In [ ]:
plt.figure(num=1, figsize=(12,5))
plt.hist(data['ask_v'], bins=100, color='tomato', edgecolor='darkred')
plt.title('Volume (Ask) distribution', size=18)
plt.ylabel('Numbers', size=14)
plt.xlabel('Volume (Ask)', size=14)
plt.show()

In [ ]:
plt.figure(num=1, figsize=(14,7))
plt.hist(data['gap_c'], bins=100, color='sandybrown', edgecolor='saddlebrown')
plt.title('Spread (Ask - Bid) of the Close price', size=16)
plt.ylabel('Numbers', size=12)
plt.xlabel('Spread, $', size=12)
plt.show()

In [ ]:
plt.figure(figsize=(15,4))
plt.plot(data['bid_v'], color='darkorange')
plt.title('Silver Bid volume, 2023', size=16)
plt.xlabel('N 5-minute bars')
plt.ylabel('Volume (Bid)')
plt.show()

In [ ]:
data['ts_prev'] = data['ts'].shift(1)

In [ ]:
data['ts_gap'] = data['ts'] - data['ts_prev']

In [ ]:
data.loc[data.index[0], 'ts_gap'] = "2 days 01:05:00"   # CHANGE: 2 days 01:(bar minutes). The first row starts a day and a week
data['ts_gap'].iloc[0]

In [ ]:
data[['ts_gap']][data['ts_gap'] > '0 days 00:05:00'].value_counts(sort=False)   # CHANGE: the bar size

In [ ]:
data['day_start'] = 0
data['week_start'] = 0

In [ ]:
data.loc[data['ts_gap'] > '0 days 00:05:00', 'day_start'] = 1   # CHANGE: the bar size

In [ ]:
data.loc[data['ts_gap'] > '1 days 00:00:00', 'week_start'] = 1

In [ ]:
print(data['day_start'].sum(), data['week_start'].sum())      # Silver 5-minute files: 128 and 26

In [ ]:
data2 = data.drop(['ts', 'ts_prev', 'ts_gap'], axis=1)

In [ ]:
data2['target_hi'] = data2['bid_h']
data2['target_lo'] = data2['bid_l'] + data2['gap_l']

## Stage 3: Normalisation (10 marks)
Prices: `(x - min) / (max - min)`. Volumes: `x / max volume`. Spreads (deltas): `(max - x) / (max - min)`, so a small spread gives a number close to 1. **The min and max come from the training part only**, so the model never sees the future.

**What you can change:** `0.8` in the `n_train` line. If the question asks for another normalisation (for example prices ÷ 100), change only the price formulas.

In [ ]:
n_rows = len(data)
n_rows

In [ ]:
n_train = int(round(n_rows * 0.8, -3))   # CHANGE: 0.8
n_train

In [ ]:
train = data.iloc[:n_train]
train.shape

In [ ]:
p_hi = (train['bid_h'] + train['gap_h']).max()
p_lo = train['bid_l'].min()
print(p_hi, p_lo)

In [ ]:
v_hi = max(max(train['bid_v']), max(train['ask_v']))
v_hi

In [ ]:
g_hi = round(max(max(train['gap_o']), max(train['gap_h']), max(train['gap_l']), max(train['gap_c'])), 3)
g_lo = round(min(min(train['gap_o']), min(train['gap_h']), min(train['gap_l']), min(train['gap_c'])), 3)
print(g_hi, g_lo)

In [ ]:
vg_hi = train['vol_gap'].max()
vg_lo = train['vol_gap'].min()
vga_hi = train['vol_gap_abs'].max()
vga_lo = train['vol_gap_abs'].min()
print(vg_hi, vg_lo, vga_hi, vga_lo)

In [ ]:
data2['bid_o'] = (data2['bid_o'] - p_lo) / (p_hi - p_lo)
data2['bid_h'] = (data2['bid_h'] - p_lo) / (p_hi - p_lo)
data2['bid_l'] = (data2['bid_l'] - p_lo) / (p_hi - p_lo)
data2['bid_c'] = (data2['bid_c'] - p_lo) / (p_hi - p_lo)
data2['target_hi'] = (data2['target_hi'] - p_lo) / (p_hi - p_lo)
data2['target_lo'] = (data2['target_lo'] - p_lo) / (p_hi - p_lo)

In [ ]:
data2['ask_v'] = data2['ask_v'] / v_hi
data2['bid_v'] = data2['bid_v'] / v_hi

In [ ]:
data2['vol_gap'] = (data2['vol_gap'] - vg_lo) / (vg_hi - vg_lo)

In [ ]:
data2['vol_gap_abs'] = data2['vol_gap_abs'] / vga_hi

In [ ]:
data2['gap_o'] = (g_hi - data2['gap_o']) / (g_hi - g_lo)
data2['gap_h'] = (g_hi - data2['gap_h']) / (g_hi - g_lo)
data2['gap_l'] = (g_hi - data2['gap_l']) / (g_hi - g_lo)
data2['gap_c'] = (g_hi - data2['gap_c']) / (g_hi - g_lo)

In [ ]:
data2.head()

In [ ]:
# normalisation check
pd.set_option('display.max_columns', None)
cols = ['bid_o', 'bid_h', 'bid_l', 'bid_c', 'bid_v', 'ask_v',
        'vol_gap', 'vol_gap_abs', 'gap_o', 'gap_h', 'gap_l', 'gap_c']
print(data2.iloc[:n_train][cols].describe().loc[['min', 'max']].round(3))   # training part: inside 0 to 1
print(data2.iloc[n_train:][cols].describe().loc[['min', 'max']].round(3))   # test part: may go slightly outside

back = data2['bid_c'] * (p_hi - p_lo) + p_lo
print((back - data['bid_c']).abs().max())                                  # must be about 0

In [ ]:
cols_float = ['bid_o', 'bid_h', 'bid_l', 'bid_c',
                 'bid_v', 'ask_v', 'vol_gap', 'vol_gap_abs',
                 'gap_o', 'gap_h', 'gap_l', 'gap_c',
                 'target_hi', 'target_lo']

cols_int = ['day_start', 'week_start']

In [ ]:
for column in cols_float:
    data2[column] = pd.to_numeric(data2[column], downcast='float')

for column in cols_int:
    data2[column] = pd.to_numeric(data2[column], downcast='integer')

data2.info()

In [ ]:
file_obj1 = open('silver_5min_normalised.csv', 'w')
data2.to_csv('silver_5min_normalised.csv', encoding='utf-8', index=False)
file_obj1.close()

## Stage 4: Shift the min and max prices, inputs and outputs, 3D tensor, split (10 marks)
The answers are the **highest `bid_h` and lowest `ask_l` over the next `WINDOW` bars** (25 minutes here). Each example is a window of the last `N` bars.

**What you can change:** the `5` in `window_size=5` and in **both** `range(5)` loops (the bars ahead = minutes ahead ÷ bar minutes, here 25 ÷ 5 = 5), `N = 50`, `n_small = 30000` and `test_size=0.1`. Change the three 5s **together**.

**Run the delete-last-rows cells only once.** `shuffle=False` stays for a time series.

In [ ]:
data = pd.read_csv('silver_5min_normalised.csv', low_memory=False, sep=',')
data.head()

In [ ]:
for column in cols_float:
    data[column] = pd.to_numeric(data[column], downcast='float')

for column in cols_int:
    data[column] = pd.to_numeric(data[column], downcast='integer')

data.info()

In [ ]:
feat_df = data.drop(['target_hi', 'target_lo'], axis=1)

In [ ]:
ans_df = data[['target_hi', 'target_lo']]

In [ ]:
ans_df = ans_df.shift(-1)

In [ ]:
# 5 bars x 5 minutes = the next 25 minutes
indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=5)   # CHANGE: bars ahead = minutes ahead / bar minutes

ans_df['target_hi_next'] = ans_df['target_hi'].rolling(window=indexer).max()
ans_df['target_lo_next'] = ans_df['target_lo'].rolling(window=indexer).min()

In [ ]:
ans_df = ans_df.drop(['target_hi', 'target_lo'], axis=1)

In [ ]:
for i in range(5):   # CHANGE: same number as window_size
    ans_df.drop(ans_df.shape[0] - 1, axis=0, inplace=True)

In [ ]:
for i in range(5):   # CHANGE: same number as window_size
    feat_df.drop(feat_df.shape[0] - 1, axis=0, inplace=True)

In [ ]:
feat_np = np.array(feat_df)

with np.printoptions(precision=4):
    print("feat_np:")
    print(feat_np[:2, :])
    print('\n')
    print(feat_np[-2:, :])

print("numpy size: = ", feat_np.shape)
print("type: ", feat_np.dtype)

In [ ]:
ans_np = np.array(ans_df)

print("ans_np:")
print(ans_np[:3, :])
print('\n')
print(ans_np[-3:, :])

print("numpy size: = ", ans_np.shape)
print("type: ", ans_np.dtype)
print(np.isnan(ans_np).sum(), 'missing values in ans_np (must be 0)')

In [ ]:
from tqdm import tqdm

In [ ]:
n_small = 30000   # CHANGE: how many rows are used
N = 50   # CHANGE: how many past bars the model looks at
L = n_small - N
t = feat_np.shape[1]
print('Size of a three-dimensional inputs tensor: ', L, N, t)

In [ ]:
tensor_x = np.zeros((L, N, t), dtype='float32')

In [ ]:
Y = np.zeros((L, 2), dtype='float32')

In [ ]:
print('L = n_small - N = ', n_small - N, L)

for k in tqdm(range(L)):
    tensor_x[k, :, :] = feat_np[k:k+N, :]

    Y[k, 0] = ans_np[k+N-1, 0]

    Y[k, 1] = ans_np[k+N-1, 1]

print(k)
print("Numpy size:        ", tensor_x.shape, '\n')

print(Y[:5, :])
print(Y[-10:, :])
print("Numpy size:         ", Y.shape)

In [ ]:
import random

pp = random.randint(50, L - 20)      # a random place in the middle of tensor_x
print('random int = ', pp)

print('tensor_x:', '\t\t', 'Y:')
print('bid_h', '\t\t', 'target_hi')

for i in range(15):
    print(tensor_x[pp+i, N-1, 1], '\t\t', Y[pp+i, 0])

In [ ]:
filled = (np.abs(tensor_x).sum(axis=(1, 2)) > 0).sum()
print(filled, L)                     # the two numbers must be equal

In [ ]:
data = []
feat_df = []
ans_df = []
feat_np = []
ans_np = []

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(tensor_x, Y, test_size=0.1,   # CHANGE: share kept for testing
                                                    shuffle=False, stratify=None, random_state=101)

In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)
print(np.isnan(X_train).sum(), np.isnan(y_train).sum(), np.isnan(X_test).sum(), np.isnan(y_test).sum())   # all must be 0

## Stage 5: Neural network architecture (10 marks)
**What you can change:** `256` and `128` (neurons), add or remove a `Dense` line, `batch_size=10`, `epochs=20`, `validation_split=0.2`. `input_shape=(N, t)` reads the numbers from your data.

Expected `summary()` total: **212,610** parameters.

In [ ]:
model = keras.Sequential([

    keras.layers.Flatten(input_shape=(N, t)),
    keras.layers.Dense(256, activation=tf.nn.relu, kernel_initializer="normal"),
    keras.layers.Dense(128, activation=tf.nn.relu, kernel_initializer="normal"),

    keras.layers.Dense(2)
])

In [ ]:
print(model.summary())

In [ ]:
model.compile(optimizer="adam", loss="mse", metrics=["mae"])

## Stage 6: Train the neural network (20 marks)
Expected during training: **2157/2157** per epoch. **Run the training cell once and leave its output on screen.**

In [ ]:
history = model.fit(X_train, y_train, batch_size=10, epochs=20, validation_split=0.2, verbose=1)

## Stage 7: Graph of the training process (10 marks)
**What you can change:** the colours and titles. `'loss'` is the MSE and `'mae'` is the MAE.

In [ ]:
history_dict = history.history

mse_values = history_dict['loss']
val_mse_values = history_dict['val_loss']

epochs = range(1, len(mse_values) + 1)
plt.figure(num=1, figsize=(15,7))
plt.plot(epochs, mse_values, 'o', color='firebrick', label='Training Mean Squared Error (MSE)')
plt.plot(epochs, val_mse_values, color='navy', linewidth=2, marker='o', markeredgecolor='black', markerfacecolor='orange', label='Validation Mean Squared Error (MSE)')
plt.title('MLP: MSE during training', size=22)
plt.xlabel('Epochs', size=18)
plt.ylabel('Mean Squared Error (MSE)', size=18)
plt.grid(True, color='lightgray', linestyle='--')
plt.legend()
plt.show()

In [ ]:
mean_absolute_error_values = history_dict['mae']
val_mean_absolute_error_values = history_dict['val_mae']

epochs = range(1, len(mean_absolute_error_values) + 1)
plt.figure(num=1, figsize=(15,7))
plt.plot(epochs, mean_absolute_error_values, 'o', color='firebrick', label='Training Mean Absolute Error (MAE)')
plt.plot(epochs, val_mean_absolute_error_values, color='navy', linewidth=2, marker='o', markeredgecolor='black', markerfacecolor='orange', label='Validation Mean Absolute Error (MAE)')
plt.title('MLP: MAE during training', size=22)
plt.xlabel('Epochs', size=18)
plt.ylabel('Mean Absolute Error (MAE)', size=18)
plt.grid(True, color='lightgray', linestyle='--')
plt.legend()
plt.show()

## Stage 8: MSE and MAE on the test data (10 marks)
The MAE in dollars is the normalised MAE times `(p_hi - p_lo)`.

In [ ]:
mse, mae = model.evaluate(X_test, y_test, verbose=1)

In [ ]:
print("Mean absolute error: %.5f" % mae)
print("Mean squared error: %.5f" % mse)
print("Mean absolute error in dollars: %.4f" % (mae * (p_hi - p_lo)))

## Stage 9: Additional graphs and predicted prices (10 marks)
**What you can change:** `N1` (first row), `LL` (how many rows), the colours and the titles. `N1 + LL` must be below the number of test rows.

In [ ]:
pred = model.predict(X_test)
print(pred)

In [ ]:
N1 = 2500
LL = 500
N2 = N1 + LL

fig = plt.figure(figsize=(15,7))

ax1 = fig.add_subplot(111)
line1, = ax1.plot(y_test[N1:N2, 0], label='High Bid', color='firebrick')
line2, = ax1.plot(pred[N1:N2, 0], label='Predicted High Bid', color='navy')
line3, = ax1.plot(y_test[N1:N2, 1], label='Low Ask', color='darkorange')
line4, = ax1.plot(pred[N1:N2, 1], label='Predicted Low Ask', color='teal')

plt.title('MLP: Real & Predicted Prices, next 25 minutes', size=22)
plt.ylabel('SILVER Prices (normalised)', size=18)
plt.xlabel('N 5-minute bars', size=18)
plt.grid(True, color='lightgray', linestyle='--')
plt.legend(loc='upper right')

plt.show()

In [ ]:
N1 = 0
LL = 1500
N2 = N1 + LL

fig = plt.figure(figsize=(15,7))

ax1 = fig.add_subplot(111)

a = (pred[N1:N2, 0] - y_test[N1:N2, 0]) * (p_hi - p_lo)
b = (pred[N1:N2, 1] - y_test[N1:N2, 1]) * (p_hi - p_lo)

line1, = ax1.plot(a, label='Predicted High Bid Error', color='firebrick')
line2, = ax1.plot(b, label='Predicted Low Ask Error', color='navy')

plt.title('MLP: Error SILVER price Chart', size=22)
plt.ylabel('Predicted Error SILVER Prices ($)', size=18)
plt.xlabel('N 5-minute bars', size=18)
plt.grid(True, color='lightgray', linestyle='--')
plt.legend(loc='upper right')

plt.show()

In [ ]:
# predicted against real (points on the red line = perfect prediction)
plt.figure(figsize=(8,8))
plt.scatter(y_test[:, 0], pred[:, 0], s=5, color='darkorange', label='High Bid')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', label='Perfect prediction')
plt.title('MLP: Predicted vs Real High Bid', size=18)
plt.xlabel('Real (normalised)', size=14)
plt.ylabel('Predicted (normalised)', size=14)
plt.legend()
plt.show()

In [ ]:
# histogram of the absolute errors in dollars
abs_error = np.abs(pred - y_test) * (p_hi - p_lo)
plt.figure(figsize=(14,7))
plt.hist(abs_error.flatten(), bins=100, color='sandybrown', edgecolor='saddlebrown')
plt.title('MLP: Histogram of absolute errors', size=18)
plt.xlabel('Absolute error, $', size=14)
plt.ylabel('Numbers', size=14)
plt.show()

In [ ]:
# predicted prices in dollars
pred_dollars = pred * (p_hi - p_lo) + p_lo
real_dollars = y_test * (p_hi - p_lo) + p_lo

result = pd.DataFrame({'Predicted max High Bid ($)': pred_dollars[:, 0],
                       'Real max High Bid ($)': real_dollars[:, 0],
                       'Predicted min Low Ask ($)': pred_dollars[:, 1],
                       'Real min Low Ask ($)': real_dollars[:, 1]})
result.tail(5).round(3)

## Analysis (write this after the final run, with your own numbers)
**Model:** MLP. It reads the last ___ five-minute bars and predicts the highest Bid high and the lowest Ask low over the next ___ minutes.

**Result:** test MSE = ___ and test MAE = ___ (normalised), which is about $___ on a silver price near $___.

**Training:** the training and validation curves ___ (both fall / validation rises), so the model ___ overfit.

**Prediction:** the predicted lines ___ the real prices. The error chart shows larger mistakes during ___.

**Limit:** only ___ months of data were used, so the model should be tested on other periods.